In [2]:
import pandas as pd 

In [3]:
dt = pd.read_csv(r'C:\Users\Utkarsh Bachhav\Downloads\OneDrive\Desktop\Python_ML\Week_18\Day_01\IMDB Dataset.csv')

In [4]:
dt.shape 

(50000, 2)

In [5]:
dt.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
dt.drop_duplicates(inplace=True) 

In [7]:
dt.shape

(49582, 2)

# Pre-Processing

### 1) converting to lowercase

In [8]:
dt['review'] = dt['review'].str.lower() 

### 2) Remove the urls

In [9]:
import re 

def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text 
dt['review'] = dt['review'].apply(remove_urls)

### 3) Removing the punctuations

In [10]:
def remove_punctuations(text):
    text = re.sub(r'[^A-Za-z0-9\s]', "", text)
    return text

dt['review'] = dt['review'].apply(remove_punctuations)

### 4) Remocing HTML

In [11]:
def remove_html(text):
    text = re.sub(r'<.*?>', "", text) 
    return text 
    
dt['review'] = dt['review'].apply(remove_html)

### 5) Removing stopwords 

In [12]:
import nltk

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [14]:
def remove_stopwords(text): 
    tokens = word_tokenize(text) 
    stop_words = stopwords.words('english')

    for word in tokens: 
        if word in stop_words: 
            text = text.replace(word, "")
    return text  

dt['review'] = dt['review'].apply(remove_stopwords)

In [15]:
dt.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6) Stemming

In [16]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = [] 

    tokens = word_tokenize(text) 

    for token in tokens: 
        stemmed_tokens = ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words) 

dt['review'] = dt['review'].apply(stemming)
        

In [17]:
dt.head() 

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7) Encoding 

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

dt['sentiment'] = le.fit_transform(dt['sentiment'])

In [19]:
y = dt['sentiment']

In [20]:
dt.head() 

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


### 8) Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(dt['review'])


## Datasets and DataLoaders

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


In [23]:
X_train.shape

(39665, 5000)

In [24]:
X_test.shape

(9917, 5000)

In [25]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3248304 stored elements and shape (39665, 5000)>

In [26]:
X_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 808836 stored elements and shape (9917, 5000)>

In [27]:
## They are in sparese form so we have to convert it into numpr array 
X_train = X_train.toarray()
X_test = X_test.toarray()

In [28]:
import torch 
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(
    torch.from_numpy(X_train).float(), 
    torch.from_numpy(y_train.values).float()  
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(), 
    torch.from_numpy(y_test.values).float() 
    
)

In [29]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64) 

## Build the RNN

In [30]:
import torch
import torch.nn as nn 
import torch.optim as optim 

In [31]:
class RNN(nn.Module): 
    def __init__(self, input_size, hidden_size=128, num_layers = 1): 
        super().__init__() 

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer 
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        # FC layer 
        self.fc = nn.Linear(hidden_size, 1) 

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 

        out =  self.fc(out[:, -1, :])

        return out  

In [32]:
input_size = X_train.shape[1] 

model = RNN(input_size)

criterion = nn.BCELoss() 

optimizer = optim.Adam(model.parameters())

In [33]:
epochs = 10 

for epoch in range(epochs):
    model.train() 

    for Xb, yb in train_loader: 
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) 
        outputs = model(Xb) 
        outputs = torch.sigmoid(outputs.squeeze()) 

        loss = criterion(outputs, yb) 
        loss.backward() 

        optimizer.step() 

    print(f'epoch = {epoch+1}/{epochs} and loss = {loss.item()}')

epoch = 1/10 and loss = 0.17334619164466858
epoch = 2/10 and loss = 0.2928089499473572
epoch = 3/10 and loss = 0.30964329838752747
epoch = 4/10 and loss = 0.174690380692482
epoch = 5/10 and loss = 0.4345146417617798
epoch = 6/10 and loss = 0.2254353165626526
epoch = 7/10 and loss = 0.15844710171222687
epoch = 8/10 and loss = 0.33639073371887207
epoch = 9/10 and loss = 0.20814894139766693
epoch = 10/10 and loss = 0.09581870585680008


In [37]:
model.eval() 

with torch.no_grad():
    correct_values = 0 
    total_values = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        
        output = model(Xb)
        predicted = (torch.sigmoid(output.squeeze()) > 0.5).float() 

        total_values += yb.size(0) 
        correct_values += (predicted == yb).sum().item() 


    print(f' accuracy = {correct_values/total_values * 100}')

        

 accuracy = 85.89291116265
